# 玉藻前文章配图生成 - Google Colab 版（修复脸部版）
使用免费 T4 GPU 生成 18 张 16:9 图片，**已加入脸部优化和修复**

## 使用说明
1. 点击上方 "复制到云端硬盘" 按钮
2. 确保右上角显示 "RAM: 12.7 GB, GPU: T4" (免费版)
3. 依次运行下面的单元格
4. 生成的图片会打包下载，**人物脸部已修复**

In [ ]:
# 第一步：安装依赖
!pip install -q diffusers transformers accelerate safetensors torch
print('依赖安装完成')

In [ ]:
# 第二步：检查 GPU
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# 第三步：加载 Stable Diffusion 1.5 模型（适合 Colab 免费版）
from diffusers import StableDiffusionPipeline
import torch

print('正在加载模型...')
pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()  # 节省显存
print('模型加载完成！')

In [ ]:
# 第四步：定义 18 个小节的提示词（**已加入脸部优化**）
# 脸部优化前缀：强制模型关注五官细节
face_optim_prefix = "(masterpiece, best quality:1.2), (extremely detailed face:1.3), (beautiful eyes:1.2), (detailed nose and lips:1.1), "

# 负面提示词：专门防止脸部畸形
negative_prompt = 'low quality, blurry, distorted, ugly, bad anatomy, watermark, signature, text, photorealistic, deformed face, bad face, missing eyes, ugly face, blurry face, bad anatomy face, extra digits, missing fingers'

prompts = [
    # 一节：妖狐的三国流转
    ('1.1_印度华阳天', face_optim_prefix + 'Ancient Indian palace, beautiful fox spirit disguised as court lady Huayangtian, golden robes, nine tails visible in shadow, Indian architecture, mysterious atmosphere, ukiyo-e style'),
    ('1.2_中国妲己', face_optim_prefix + 'Chinese Shang dynasty palace, stunning concubine Daji, silk robes, nine fox tails shadow, bronze vessels, oracle bones, Chinese painting style'),
    ('1.3_日本玉藻前', face_optim_prefix + 'Japanese Heian period court, beautiful Tamamo-no-Mae in elegant kimono, entering palace, cherry blossoms, ukiyo-e style'),
    
    # 二节：宫廷中的妖狐魅影
    ('2.1_入宫得宠', face_optim_prefix + 'Tamamo-no-Mae playing koto, Heian court ladies admiring her, palace interior, golden screens, traditional Japanese painting'),
    ('2.2_天皇病重', face_optim_prefix + 'Sick Emperor Toba in bed, dark palace room, mysterious fox shadow on wall, dramatic lighting, ukiyo-e style'),
    ('2.3_阴阳师怀疑', face_optim_prefix + 'Yin-Yang master Abe Yasunari meditating, seeing nine-tailed fox shadow, moonlight, Edo period style'),
    
    # 三节：真身败露与逃亡
    ('3.1_识破妖身', face_optim_prefix + 'Yin-Yang masters casting spells, nine-tailed fox spirit revealed, magical circles, Japanese mythology'),
    ('3.2_天皇的震惊', face_optim_prefix + 'Emperor Toba shocked, mirror reflection showing fox ears, palace room, dramatic lighting'),
    ('3.3_那须野的藏身', face_optim_prefix + 'Abandoned mansion in Nasu field, moonlight, mysterious atmosphere, Japanese countryside'),
    
    # 四节：那须野的讨伐之战
    ('4.1_三浦介与上总介', face_optim_prefix + 'Japanese samurai warriors preparing for battle, traditional armor, Nasu field, ukiyo-e style'),
    ('4.2_激战那须野', face_optim_prefix + 'Epic battle, nine-tailed fox giant form, samurai fighting, dynamic action, Japanese mythology'),
    ('4.3_妖狐之死', face_optim_prefix + 'Nine-tailed fox falling, arrow in forehead, dramatic sunset, tragic scene'),
    
    # 五节：杀生石的诅咒
    ('5.1_石头诞生', face_optim_prefix + 'Giant stone formation, poisonous aura, dead vegetation, Japanese landscape'),
    ('5.2_镇魂与封印', face_optim_prefix + 'Buddhist monk chanting sutra, glowing stone, peaceful atmosphere, Japanese temple'),
    ('5.3_现代遗迹', face_optim_prefix + 'Tourists visiting Sessho-seki stone, modern Japan, historical site'),
    
    # 六节：文化影响与后世演绎
    ('6.1_文学与戏剧', face_optim_prefix + 'Traditional Japanese theater, Noh mask, scroll paintings, Tamamo-no-Mae story'),
    ('6.2_现代流行文化', face_optim_prefix + 'Anime style Tamamo-no-Mae, modern illustration, vibrant colors, manga aesthetic'),
    ('6.3_东亚妖狐文化的交融', face_optim_prefix + 'Three women India China Japan, fox spirits, cultural exchange, artistic composition'),
]

print(f'准备生成 {len(prompts)} 张图片，已加入脸部优化提示')

In [ ]:
# 第五步：生成图片（**已加入脸部修复**）
import os
import cv2
import numpy as np
from PIL import Image

# 安装面部修复依赖
!pip install -q gfpgan basicsr facexlib
from gfpgan import GFPGANer

# 初始化 GFPGAN 修复器（腾讯开源脸部修复模型）
print('正在初始化脸部修复模型...')
gfpganer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
    upscale=2,
    arch='clean',
    channel_multiplier=2,
    bg_upsampler=None
)
print('脸部修复模型加载完成！')

# 创建输出目录
os.makedirs('tamamo_images', exist_ok=True)

print('\n开始生成图片（分辨率 768x432 → 放大到 1920x1080）...\n')
for i, (name, prompt) in enumerate(prompts, 1):
    print(f'[{i}/{len(prompts)}] 生成: {name}')
    print(f'   提示词: {prompt[:60]}...')
    
    try:
        # 生成 768x432 (16:9，比之前更高清，脸部更清晰)
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=768,
            height=432,
            num_inference_steps=30,  # 增加步数，提升细节
            guidance_scale=7.5,
        ).images[0]
        
        # 面部修复：用 GFPGAN 修复脸部畸形
        input_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        _, _, restored_img = gfpganer.enhance(input_img, has_aligned=False, paste_back=True)
        restored_image = Image.fromarray(cv2.cvtColor(restored_img, cv2.COLOR_BGR2RGB))
        
        # 放大到 1920x1080
        image_large = restored_image.resize((1920, 1080), Image.LANCZOS)
        
        # 保存
        output_path = f'tamamo_images/{name}.png'
        image_large.save(output_path, 'PNG', quality=95)
        print(f'   ✓ 保存: {output_path}（已修复脸部）')
        
    except Exception as e:
        print(f'   ✗ 失败: {e}')

print('\n所有图片生成完成！')

In [ ]:
# 第六步：打包并下载
import shutil
from google.colab import files

# 打包成 zip
shutil.make_archive('tamamo_images', 'zip', 'tamamo_images')
print('打包完成: tamamo_images.zip')

# 下载
files.download('tamamo_images.zip')

In [ ]:
# (可选) 第七步：上传到 Google Drive 备份
from google.colab import drive
drive.mount('/content/drive')

import shutil
dst = '/content/drive/MyDrive/tamamo_images'
shutil.copytree('/content/tamamo_images', dst)
print(f'已备份到 Google Drive: {dst}')

In [ ]:
# （测试用）只生成第一张图片，快速验证脸部效果
print('=== 测试生成第一张：1.1_印度华阳天 ===')
test_name, test_prompt = prompts[0]
print(f'提示词: {test_prompt}')

try:
    test_image = pipe(
        prompt=test_prompt,
        negative_prompt=negative_prompt,
        width=768,
        height=432,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]
    
    # 修复脸部
    test_input = cv2.cvtColor(np.array(test_image), cv2.COLOR_RGB2BGR)
    _, _, test_restored = gfpganer.enhance(test_input, has_aligned=False, paste_back=True)
    test_restored_img = Image.fromarray(cv2.cvtColor(test_restored, cv2.COLOR_BGR2RGB))
    
    # 显示测试图
    display(test_restored_img.resize((960, 540)))
    print('测试图生成完成！如果脸部清晰，再运行上一个单元格生成全部')
    
except Exception as e:
    print(f'测试失败: {e}')